In [ ]:
import time
import pika
import sys
import numpy as np
import datetime

credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters('149.62.71.186', credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()

channel.exchange_declare(exchange='direct_logs', exchange_type='direct')

In [ ]:
while True:
    time.sleep(1)
    key = np.random.choice(['info', 'warning', 'error', 'OTHER'])
    channel.basic_publish(exchange = 'direct_logs', routing_key = key, body = str(datetime.datetime.now()))

In [ ]:
print('Waiting for logs. To exit press CTRL+C')


def callback(ch, method, properties, body):
    print("Received %r:%r" % (method.routing_key, body))


channel.basic_consume(
    queue=queue_name, on_message_callback=callback, auto_ack=True)

In [ ]:
channel.start_consuming()


## 1. Establishing the Connection
The first block sets up the "handshake" with the RabbitMQ server.
* **Credentials:** It uses a username and password (`martin`).
* **Parameters:** It points to a specific IP address (`149.62.71.186`).
* **Channel:** Opens a virtual connection (channel) where the actual data travels.

## 2. The Exchange Logic
```python
channel.exchange_declare(exchange='direct_logs', exchange_type='direct')
```
The code uses a **Direct Exchange**. In RabbitMQ, an exchange is like a mail sorter. A "direct" type means a message goes to a specific queue based on an exact match of a **Routing Key**.



---

## 3. The Producer Logic (The `while` loop)
This section acts as a log generator.
* **The Randomizer:** It picks a random severity level: `info`, `warning`, `error`, or `OTHER`.
* **The Publication:** Every second, it sends the current timestamp to the `direct_logs` exchange. 
* **The Key:** The `routing_key` is set to the random severity. This allows receivers to choose if they only want to hear about "errors" or everything.

---

## 4. The Consumer Logic (The `callback`)
The bottom half of the script is designed to listen for messages.
* **The Callback:** This function defines what to do when a message arrives (in this case, just printing it).
* **`basic_consume`:** Tells RabbitMQ to start pulling messages from a queue.
* **`start_consuming`:** An infinite loop that waits for data to arrive.
